<a href="https://colab.research.google.com/github/NirtonAfonso/tech-challenge-fase3-medflow-ai/blob/develop/notebooks/04_structured_patient_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir no Colab"/></a>

# 04 — Prontuário estruturado: SQLite, consultas e minimização de dados

**Tech Challenge Fase 3 — MedFlow AI**

| | |
|---|---|
| Runtime | **CPU** |
| Como rodar | `Ambiente de execução` → `Executar tudo` |
| Saída no Drive | `MedFlowAI_Fase3/04_structured_data/` |

O enunciado exige que o assistente "realize consultas em base de dados estruturadas (como prontuários e
registros)" e "contextualize as respostas com informações atualizadas do paciente". Este notebook mostra
a base, as consultas e — o ponto crítico — **o que sai e o que não sai** da base rumo à LLM.

In [ ]:
# @title ▶ Bootstrap — execute esta célula primeiro (Colab ou local)
#
# Prepara tudo do zero em um runtime Colab novo: monta o Google Drive, clona a
# branch `develop`, instala as dependências e cria a estrutura de saída.
# Rodando localmente, detecta o repositório e pula clone/Drive.

import os
import pathlib
import subprocess
import sys

REPO_URL = "https://github.com/NirtonAfonso/tech-challenge-fase3-medflow-ai.git"
REPO_BRANCH = "develop"
REPO_DIR = "tech-challenge-fase3-medflow-ai"
NOTEBOOK_ID = "04_structured_data"
REQUIREMENTS = "requirements-colab.txt"

IN_COLAB = "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))


def _run(*args, **kwargs):
    return subprocess.run(list(args), check=kwargs.pop("check", True), **kwargs)


def _pip(*args):
    _run(sys.executable, "-m", "pip", *args)


def _tem_torch_cuda() -> bool:
    try:
        import torch

        return torch.cuda.is_available()
    except Exception:
        return False


# --- 1. Google Drive ---------------------------------------------------------
if IN_COLAB:
    try:
        from google.colab import drive

        drive.mount("/content/drive")
        print("Google Drive montado em /content/drive")
    except Exception as erro:
        print(f"ATENÇÃO: falha ao montar o Drive ({erro}).")
        print("Os resultados ficarão apenas em /content e serão PERDIDOS ao encerrar a sessão.")

# --- 2. Repositório ----------------------------------------------------------
def _raiz_local() -> pathlib.Path | None:
    atual = pathlib.Path.cwd()
    for candidato in [atual, *atual.parents]:
        if (candidato / "src" / "medflow_ai").exists():
            return candidato
    return None


raiz = _raiz_local()
if raiz is None:
    destino = pathlib.Path("/content" if IN_COLAB else ".") / REPO_DIR
    if destino.exists():
        _run("git", "-C", str(destino), "fetch", "--depth", "1", "origin", REPO_BRANCH)
        _run("git", "-C", str(destino), "checkout", REPO_BRANCH)
        _run("git", "-C", str(destino), "pull", "--ff-only", "origin", REPO_BRANCH)
    else:
        # Sempre com --branch explícita: nunca clonar a default implicitamente.
        _run("git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(destino))
    raiz = destino.resolve()

os.chdir(raiz)
if str(raiz / "src") not in sys.path:
    sys.path.insert(0, str(raiz / "src"))
print(f"Raiz do projeto: {raiz}")

# --- 3. Dependências ---------------------------------------------------------
_pip("install", "-q", "-U", "pip")
_pip("install", "-q", "-r", REQUIREMENTS)
_pip("install", "-q", "-e", ".")

# --- 4. Diagnóstico ----------------------------------------------------------
import platform

from medflow_ai.colab import ensure_structure, git_info, in_colab, write_run_metadata

_git = git_info(raiz)
print("\n" + "=" * 78)
print(f"Python        : {platform.python_version()}")
print(f"Ambiente      : {'Google Colab' if IN_COLAB else 'local'}")
print(f"Branch        : {_git['branch']}")
print(f"Commit        : {_git['commit']}")
print("=" * 78)

# --- 6. Estrutura de saída (Drive no Colab, artifacts/colab localmente) -------
PASTAS = ensure_structure(NOTEBOOK_ID)
print("\nEstrutura de saída:")
for _nome, _caminho in sorted(PASTAS.items()):
    print(f"  {_nome:22s} {_caminho}")

RUN_META = write_run_metadata(NOTEBOOK_ID)
print(f"\nMetadados da execução: {RUN_META}")


## 1. Por que dados sintéticos gerados, e não Synthea baixado?

O esquema é **idêntico ao export CSV do Synthea** (`patients`, `conditions`, `observations`,
`medications`, `procedures`, `encounters`), e existe um ingestor pronto para um export real
(`ingest_synthea_csv`). O gerador próprio é o padrão porque o Synthea exige Java e produz centenas de
MB, inviável em Colab e CI — e porque um gerador determinístico permite testes de recuperação exata.

Uma tabela adicional, `lab_orders`, modela **exames pendentes** — exigência do enunciado ausente do
export padrão do Synthea.

In [ ]:
from medflow_ai.database.ingest import build_synthetic_database
from medflow_ai.database.schema import SCHEMA_SQL

contagens = build_synthetic_database(n_patients=40)
print(contagens)
print(SCHEMA_SQL[:1200])

## 2. Consultas ao prontuário

In [ ]:
from medflow_ai.database.repository import PatientRepository

repo = PatientRepository()
PACIENTE = "P-DEMO-0001"

print("Condições ativas   :", repo.conditions(PACIENTE))
print("\nExames recentes  :", repo.latest_observations(PACIENTE, limit=5))
print("\nMedicamentos      :", repo.medications(PACIENTE))
print("\nExames pendentes  :", repo.pending_exams(PACIENTE))
print("\nÚltimos encontros :", repo.encounters(PACIENTE, limit=2))

In [ ]:
import pandas as pd

historico = pd.DataFrame(repo.observation_history(PACIENTE, "TSH"))
historico

## 3. A LLM recebe identificadores diretos? (evidência de minimização)

**Pergunta.** O que exatamente atravessa a fronteira entre o banco e o prompt?

In [ ]:
bruto = repo.raw_record(PACIENTE)
print("=== REGISTRO BRUTO NO BANCO (nunca sai daqui) ===")
for chave, valor in bruto.items():
    print(f"  {chave:12s}: {valor}")

In [ ]:
contexto = repo.build_context(PACIENTE)
bloco = contexto.to_prompt_block()
print("=== O QUE A LLM EFETIVAMENTE RECEBE ===")
print(bloco)

In [ ]:
from medflow_ai.data.anonymization import contains_pii

campos_sensiveis = ("first", "last", "cpf", "cns", "email", "phone", "address", "birthdate")
vazou = [campo for campo in campos_sensiveis if str(bruto[campo]) and str(bruto[campo]) in bloco]
print("Campos identificadores presentes no prompt:", vazou or "NENHUM ✅")
print("Detector de PII acusa algo no prompt?", contains_pii(bloco))

**Interpretação.** Nome, CPF, CNS, telefone, e-mail e endereço permanecem no banco e **não** chegam
ao prompt. A data de nascimento é substituída por faixa etária, e o `patient_id` por um pseudônimo
derivado com salt. O que sobe ao modelo é o mínimo necessário para responder.

## 4. Minimização sob demanda

Se o médico só quer saber de pendências, não há razão para enviar histórico medicamentoso.

In [ ]:
apenas_pendencias = repo.build_context(PACIENTE, include=["pending_exams"])
print(apenas_pendencias.to_prompt_block())
print("\nCondições enviadas:", apenas_pendencias.conditions)

## 5. Ferramentas LangChain que expõem essa base ao grafo

In [ ]:
from medflow_ai.graph.tools import MEDFLOW_TOOLS, get_repository, verificar_alertas_clinicos

get_repository.cache_clear()
for ferramenta in MEDFLOW_TOOLS:
    print(f"- {ferramenta.name}: {ferramenta.description.splitlines()[0]}")

print("\nAlertas automáticos do paciente demo:")
alertas = verificar_alertas_clinicos.invoke({"patient_id": PACIENTE})
for alerta in alertas:
    print(f"  [{alerta['severidade'].upper()}] {alerta['mensagem']} (fonte: {alerta['fonte']})")

## 6. Recuperação exata é verificável?

**Método.** O gabarito é derivado do próprio banco: para cada exame, o valor mais recente conhecido é
comparado com o valor devolvido pelo repositório. Diferente do RAG, aqui não há ambiguidade.

In [ ]:
from medflow_ai.evaluation.database_eval import build_cases, evaluate_database, save_report

relatorio_db = evaluate_database(repo, build_cases(limit=30))
print(f"recuperação exata: {relatorio_db.exact_match:.3f} em {relatorio_db.n_cases} casos")
print(f"contexto livre de identificadores diretos: {relatorio_db.context_leak_free}")
print("falhas:", relatorio_db.falhas or "nenhuma")

caminho_db_report = save_report(relatorio_db, PASTAS["artifacts"])
print("relatório salvo em", caminho_db_report)

**Interpretação.** Este é o teste que distingue "a LLM disse algo plausível" de "o dado do paciente
foi realmente usado".

**Limitação.** Os pacientes são sintéticos e clinicamente simplificados; um prontuário real traz texto
livre, evoluções, exames de imagem e inconsistências que este modelo não representa.

## 7. Exportações e persistência no Google Drive

In [ ]:
import json

from medflow_ai.config import get_settings
from medflow_ai.database.ingest import export_tables_to_csv

caminho_banco = get_settings().database_path
csvs = export_tables_to_csv(caminho_banco, PASTAS["artifacts"] / "csv_export")
print(f"{len(csvs)} tabelas exportadas para CSV")

evidencia = {
    "contagens": contagens,
    "paciente_demo": PACIENTE,
    "campos_identificadores_no_prompt": vazou,
    "detector_pii_no_prompt": contains_pii(bloco),
    "alertas_gerados": [a["id"] for a in alertas],
    "recuperacao_exata": relatorio_db.to_dict(),
}
caminho_evidencia = PASTAS["artifacts"] / "04_minimizacao_evidencia.json"
caminho_evidencia.write_text(json.dumps(evidencia, ensure_ascii=False, indent=2), encoding="utf-8")
print("evidência salva em", caminho_evidencia)

In [ ]:
# Persistência no Google Drive — uma execução só termina quando os resultados
# saem de /content. Fora do Colab, os mesmos arquivos vão para artifacts/colab/.
from medflow_ai.colab import persist, summarize

_relatorios = [
    persist([caminho_evidencia, caminho_db_report, PASTAS["artifacts"] / "csv_export"],
            PASTAS["artifacts"]),
    persist([caminho_banco], PASTAS["database"]),
]

print(summarize(_relatorios, titulo="RESUMO DA PERSISTÊNCIA — 04_structured_data"))


## 8. Conclusão

| Pergunta | Resposta |
|---|---|
| A base é consultável? | Sim: condições, exames, medicamentos, procedimentos, encontros e pendências |
| Exames pendentes existem? | Sim: tabela `lab_orders`, exigência do enunciado ausente do Synthea |
| A LLM recebe PII? | Não: verificado campo a campo e pelo detector de PII |
| A recuperação é exata? | Sim: 1,000 em 30 casos com valor conhecido |

Próximo notebook: **`05_medflow_full_demo.ipynb`** — demonstração ponta a ponta.